In [46]:
import os
import re
import math
import numpy as np
import pandas as pd
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [47]:
PROJECT_DIR = os.path.dirname(os.getcwd())
data = pd.read_csv(os.path.join(PROJECT_DIR, "data", "final_data.csv"))
data = data.dropna(subset=["whole_text", "final_quality_score", "expected_quality_class"]).reset_index(drop=True)

label_map = {"Good": 0, "average": 1, "poor": 2}
data["label"] = data["expected_quality_class"].map(label_map)
data["final_quality_score"] = data["final_quality_score"].astype(float)
data.head()

,id,original_id,category,source_type,degradation_level,expected_quality_class,whole_text,final_quality_score,label
0,2,2,Finansal Hizmetler,original,none,Good,"representative: Merhaba, size nasıl yardımcı o...",56.50,0
1,3,3,Finansal Hizmetler,original,none,Good,"customer: Merhaba, BP monitörü satın almaya ça...",82.00,0
2,4,4,Finansal Hizmetler,original,none,Good,"customer: Merhaba, son siparişimde bir fatura ...",92.25,0
3,5,5,Finansal Hizmetler,original,none,Good,"representative: Merhaba, benim adım John. Size...",86.75,0
4,6,6,Finansal Hizmetler,original,none,Good,"customer: Merhaba, BrownBox'tan aldığım telefo...",90.25,0


In [48]:
bert_name = "dbmdz/bert-base-turkish-cased"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(bert_name)
bert = AutoModel.from_pretrained(bert_name).to(device).eval()

@torch.no_grad()
def cls_embeddings(texts, batch_size=16, max_length=256):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i + batch_size]]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(device)
        cls = bert(**enc).last_hidden_state[:, 0, :]
        vecs.append(cls.cpu().numpy())
    return np.vstack(vecs)

X_all = cls_embeddings(data["whole_text"].tolist())
print(X_all.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(2695, 768)


In [49]:
y_score = data["final_quality_score"].values
y_class = data["label"].values

idx_train, idx_test = train_test_split(
    np.arange(len(data)), test_size=0.2, random_state=17, stratify=y_class
)

X_train, X_test = X_all[idx_train], X_all[idx_test]
y_train, y_test = y_score[idx_train], y_score[idx_test]
y_train_cls = y_class[idx_train]
train_df = data.iloc[idx_train].reset_index(drop=True)
test_df = data.iloc[idx_test].reset_index(drop=True)
print(X_train.shape, X_test.shape)

(2156, 768) (539, 768)


In [50]:
def evaluation(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"\n{model_name}")
    print("MAE :", round(mae, 4))
    print("RMSE:", round(rmse, 4))
    print("R2  :", round(r2, 4))
    return {"model": model_name, "mae": mae, "rmse": rmse, "r2": r2}

results = []

In [51]:
# MaxEnt = multinomial logistic regression on BERT [CLS] features.
# Class probabilities -> 0-100 score via anchor scores (Good=90, average=55, poor=15).
score_anchors = np.array([90.0, 55.0, 15.0])

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

maxent = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs",
                            class_weight="balanced", random_state=17)
maxent.fit(X_train_sc, y_train_cls)

maxent_scores = np.clip(maxent.predict_proba(X_test_sc) @ score_anchors, 0, 100)
results.append(evaluation(y_test, maxent_scores, "MaxEnt Classifier (BERT features)"))


MaxEnt Classifier (BERT features)
MAE : 15.5448
RMSE: 21.4983
R2  : 0.4029


In [52]:
# LLR Monitor: bigram language models on agent utterances from Good vs Poor calls.
TURN_RE = re.compile(r"(representative|customer):\s*", re.IGNORECASE)

def agent_utterances(text):
    parts = TURN_RE.split(str(text).strip())
    utts, i = [], 1
    while i < len(parts) - 1:
        if parts[i].strip().lower() == "representative" and parts[i + 1].strip():
            utts.append(parts[i + 1].strip())
        i += 2
    return utts

class BigramLM:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.bigrams = Counter()
        self.unigrams = Counter()
        self.vocab = set()

    def _tokens(self, text):
        return ["<s>"] + re.findall(r"\w+", text.lower()) + ["</s>"]

    def train(self, utterances):
        for utt in utterances:
            toks = self._tokens(utt)
            self.vocab.update(toks)
            for a, b in zip(toks, toks[1:]):
                self.bigrams[(a, b)] += 1
                self.unigrams[a] += 1

    def log_prob(self, text):
        toks = self._tokens(text)
        V = max(len(self.vocab), 1)
        lp = 0.0
        for a, b in zip(toks, toks[1:]):
            lp += math.log((self.bigrams[(a, b)] + self.alpha) /
                           (self.unigrams[a] + self.alpha * V))
        return lp

def collect(df_subset):
    utts = []
    for t in df_subset["whole_text"]:
        utts += agent_utterances(t)
    return utts

lm_good = BigramLM(); lm_good.train(collect(train_df[train_df["label"] == 0]))
lm_bad = BigramLM(); lm_bad.train(collect(train_df[train_df["label"] == 2]))

def llr_score(text):
    cum = sum(lm_good.log_prob(u) - lm_bad.log_prob(u) for u in agent_utterances(text))
    return (max(-50.0, min(50.0, cum)) + 50.0)   # normalize to 0-100

llr_scores = np.array([llr_score(t) for t in test_df["whole_text"]])
results.append(evaluation(y_test, llr_scores, "LLR Monitor"))


LLR Monitor
MAE : 28.7841
RMSE: 35.6238
R2  : -0.6396


In [53]:
# Hybrid: final_score = 0.70 * MaxEnt + 0.30 * LLR
hybrid_scores = np.clip(0.70 * maxent_scores + 0.30 * llr_scores, 0, 100)
results.append(evaluation(y_test, hybrid_scores, "MaxEnt + LLR Hybrid"))


MaxEnt + LLR Hybrid
MAE : 15.4876
RMSE: 20.4817
R2  : 0.458


In [54]:
# Combine with previously saved baseline + BERTurk results
prev = pd.read_csv(os.path.join(PROJECT_DIR, "data", "model_comparison_results.csv"))
prev = prev.drop(columns=["Unnamed: 0"], errors="ignore")

comparison_df = pd.concat([prev, pd.DataFrame(results)], ignore_index=True)
comparison_df = comparison_df.sort_values("mae").reset_index(drop=True)
comparison_df

,model,mae,rmse,r2
0,BERTurk Fine-Tuned Regression,7.347677,10.207565,0.876096
1,TF-IDF + Ridge Regression,9.827400,12.953900,0.784900
2,CountVectorizer + Ridge Regression,11.049861,14.366452,0.754563
3,Sentence-BERT + Ridge Regression,13.480433,17.232590,0.646864
4,MaxEnt + LLR Hybrid,15.487590,20.481651,0.458010
5,MaxEnt Classifier (BERT features),15.544840,21.498349,0.402866
6,LLR Monitor,28.784150,35.623838,-0.639619


In [55]:
comparison_df.to_csv(
    "model_comparison_final_results.csv",
    index=False,
    encoding="utf-8-sig"
)